In [1]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt

In [2]:
def read_ops(list_session_data_path):
    list_ops = []
    for session_data_path in list_session_data_path:
        ops = np.load(
            os.path.join(session_data_path, 'ops.npy'),
            allow_pickle=True).item()
        ops['save_path0'] = os.path.join(session_data_path)
        list_ops.append(ops)
    return list_ops

In [3]:
list_session_data_path = [
    r'F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250814_2afc-584',
    # '/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/2p/YH24LG_CRBL_lobulev_20250811_2afc-580',
    # '/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/2p/YH24LG_CRBL_lobulev_20250808_2afc-577',
    # '/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/2p/YH24LG_CRBL_lobulev_20250805_2afc-568',
]

list_ops = read_ops(list_session_data_path)

In [4]:
# create a numpy memmap from an h5py dataset.
def create_memmap(data, dtype, mmap_path):
    memmap_arr = np.memmap(mmap_path, dtype=dtype, mode='w+', shape=data.shape)
    memmap_arr[:] = data[...]
    return memmap_arr

# create folder for h5 data.
def get_memmap_path(ops, h5_file_name):
    mm_folder_name, _ = os.path.splitext(h5_file_name)
    if not os.path.exists(os.path.join(ops['save_path0'], 'memmap', mm_folder_name)):
        os.makedirs(os.path.join(ops['save_path0'], 'memmap', mm_folder_name))
    mm_path = os.path.join(ops['save_path0'], 'memmap', mm_folder_name)
    file_path = os.path.join(ops['save_path0'], h5_file_name)
    return mm_path, file_path

####################################################################
# read masks.
def read_masks(ops):
    mm_path, file_path = get_memmap_path(ops, 'masks.h5')
    with h5py.File(file_path, 'r') as f:
        labels     = create_memmap(f['labels'],     'int8',    os.path.join(mm_path, 'labels.mmap'))
        masks      = create_memmap(f['masks_func'], 'float32', os.path.join(mm_path, 'masks_func.mmap'))
        mean_func  = create_memmap(f['mean_func'],  'float32', os.path.join(mm_path, 'mean_func.mmap'))
        max_func   = create_memmap(f['max_func'],   'float32', os.path.join(mm_path, 'max_func.mmap'))
        mean_anat  = create_memmap(f['mean_anat'],  'float32', os.path.join(mm_path, 'mean_anat.mmap')) if ops['nchannels'] == 2 else None
        masks_anat = create_memmap(f['masks_anat'], 'float32', os.path.join(mm_path, 'masks_anat.mmap')) if ops['nchannels'] == 2 else None
    return [labels, masks, mean_func, max_func, mean_anat, masks_anat]

In [5]:
# read dff traces.
def read_dff(ops):
    mm_path, file_path = get_memmap_path(ops, 'dff.h5')
    with h5py.File(file_path, 'r') as f:
        dff = create_memmap(f['dff'], 'float32', os.path.join(mm_path, 'dff.mmap'))
    return dff

In [6]:
# read raw_voltages.h5.
def read_raw_voltages(ops):
    mm_path, file_path = get_memmap_path(ops, 'raw_voltages.h5')
    with h5py.File(file_path, 'r') as f:
        vol_time     = create_memmap(f['raw']['vol_time'],     'float32', os.path.join(mm_path, 'vol_time.mmap'))
        vol_start    = create_memmap(f['raw']['vol_start'],    'int8',    os.path.join(mm_path, 'vol_start.mmap'))
        vol_stim_vis = create_memmap(f['raw']['vol_stim_vis'], 'int8',    os.path.join(mm_path, 'vol_stim_vis.mmap'))
        vol_hifi     = create_memmap(f['raw']['vol_hifi'],     'int8',    os.path.join(mm_path, 'vol_hifi.mmap'))
        vol_img      = create_memmap(f['raw']['vol_img'],      'int8',    os.path.join(mm_path, 'vol_img.mmap'))
        vol_stim_aud = create_memmap(f['raw']['vol_stim_aud'], 'float32', os.path.join(mm_path, 'vol_stim_aud.mmap'))
        vol_flir     = create_memmap(f['raw']['vol_flir'],     'int8',    os.path.join(mm_path, 'vol_flir.mmap'))
        vol_pmt      = create_memmap(f['raw']['vol_pmt'],      'int8',    os.path.join(mm_path, 'vol_pmt.mmap'))
        vol_led      = create_memmap(f['raw']['vol_led'],      'int8',    os.path.join(mm_path, 'vol_led.mmap'))
    return [vol_time, vol_start, vol_stim_vis, vol_img,
            vol_hifi, vol_stim_aud, vol_flir,
            vol_pmt, vol_led]

In [7]:
import scipy.io as sio
import pandas as pd
import numpy as np
import os

def read_bpod_mat_data(ops, session_start_time, vol_time=None, vol_start=None, vol_stim_vis=None):
    """
    Read bpod session data with optional volume/imaging synchronization.
    
    Parameters:
    - ops: options dict with 'save_path0'
    - session_start_time: reference time for alignment
    - vol_time: volume timestamps (optional, for imaging sync)
    - vol_start: volume start signal (optional, for imaging sync)
    - vol_stim_vis: stimulus visibility signal (optional, for imaging sync)
    """
    
    def _check_keys(d):
        for key in d:
            if isinstance(d[key], sio.matlab.mat_struct):
                d[key] = _todict(d[key])
        return d

    def _todict(matobj):
        d = {}
        for strg in matobj._fieldnames:
            elem = matobj.__dict__[strg]
            if isinstance(elem, sio.matlab.mat_struct):
                d[strg] = _todict(elem)
            elif isinstance(elem, np.ndarray):
                d[strg] = _tolist(elem)
            else:
                d[strg] = elem
        return d

    def _tolist(ndarray):
        elem_list = []
        for sub_elem in ndarray:
            if isinstance(sub_elem, sio.matlab.mat_struct):
                elem_list.append(_todict(sub_elem))
            elif isinstance(sub_elem, np.ndarray):
                elem_list.append(_tolist(sub_elem))
            else:
                elem_list.append(sub_elem)
        return elem_list

    def states_labeling(trial_states):
        if 'Punish' in trial_states.keys() and not np.isnan(trial_states['Punish'][0]):
            outcome = 'punish'
        elif 'Reward' in trial_states.keys() and not np.isnan(trial_states['Reward'][0]):
            outcome = 'reward'
        elif 'PunishNaive' in trial_states.keys() and not np.isnan(trial_states['PunishNaive'][0]):
            outcome = 'naive_punish'
        elif 'RewardNaive' in trial_states.keys() and not np.isnan(trial_states['RewardNaive'][0]):
            outcome = 'naive_reward'
        elif 'DidNotChoose' in trial_states.keys() and not np.isnan(trial_states['DidNotChoose'][0]):
            outcome = 'no_choose'
        else:
            outcome = 'other'
        return outcome

    def get_state(trial_state_dict, target_state, trial_start):
        if target_state in trial_state_dict:
            time_state = 1000 * np.array(trial_state_dict[target_state]) + trial_start
        else:
            time_state = np.array([np.nan, np.nan])
        return time_state

    # Read raw data
    raw = sio.loadmat(
        os.path.join(ops['save_path0'], 'bpod_session_data.mat'),
        struct_as_record=False, squeeze_me=True)
    raw = _check_keys(raw)['SessionData']
    trial_labels = dict()
    n_trials = raw['nTrials']
    trial_states = [raw['RawEvents']['Trial'][ti]['States'] for ti in range(n_trials)]
    trial_events = [raw['RawEvents']['Trial'][ti]['Events'] for ti in range(n_trials)]

    # Trial start and end timestamps
    if vol_start is not None and vol_time is not None:
        # Use volume timing for trial start detection
        trials_start = np.where(np.diff(vol_start == 1))[0]
        trials_start = trials_start[::2]
        trial_labels['time_trial_start'] = vol_time[trials_start]
    else:
        # Use bpod timestamps directly
        trial_labels['time_trial_start'] = 1000 * np.array(raw['TrialStartTimestamp']).reshape(-1)

    trial_labels['time_trial_end'] = 1000 * np.array(raw['TrialEndTimestamp']).reshape(-1)

    # Correct timestamps starting from session start
    trial_labels['time_trial_end'] = trial_labels['time_trial_end'] - trial_labels['time_trial_start'][0] + session_start_time
    trial_labels['time_trial_start'] = trial_labels['time_trial_start'] - trial_labels['time_trial_start'][0] + session_start_time

    # Trial target
    trial_labels['trial_type'] = np.array(raw['TrialTypes']).reshape(-1) - 1
    trial_labels['block_type'] = np.array(raw['BlockTypes']).reshape(-1)

    # Trial outcomes
    trial_labels['outcome'] = np.array([states_labeling(ts) for ts in trial_states], dtype='object')

    # Average ISI
    try:
        mean_short_isi = np.array(raw['TrialSettings'][0]['GUI']['ISIShortMean_s']).reshape(-1)
        mean_long_isi = np.array(raw['TrialSettings'][0]['GUI']['ISILongMean_s']).reshape(-1)
    except Exception:
        mean_short_isi = np.nan
        mean_long_isi = np.nan

    # Trial state timings
    trial_labels['state_window_choice'] = np.array([
        get_state(trial_states[ti], 'WindowChoice', trial_labels['time_trial_start'][ti])
        for ti in range(n_trials)] + ['yicong_forever'], dtype='object')[:-1]
    trial_labels['state_reward'] = np.array([
        get_state(trial_states[ti], 'Reward', trial_labels['time_trial_start'][ti])
        for ti in range(n_trials)] + ['yicong_forever'], dtype='object')[:-1]
    trial_labels['state_punish'] = np.array([
        get_state(trial_states[ti], 'Punish', trial_labels['time_trial_start'][ti])
        for ti in range(n_trials)] + ['yicong_forever'], dtype='object')[:-1]

    # Stimulus timing
    trial_isi = []
    trial_stim_seq = []
    expected_stim = []

    if vol_stim_vis is not None and vol_time is not None:
        # Use volume-based stimulus timing
        indices = np.where(np.diff(vol_stim_vis == 1))[0]
        stim_times = vol_time[indices]
        
        for ti in range(n_trials):
            trial_indices = indices[ti * 4:(ti + 1) * 4]
            if len(trial_indices) == 4:
                trial_times = stim_times[ti * 4:(ti + 1) * 4]
                stim_seq = np.array([[trial_times[0], trial_times[1]],
                                    [trial_times[2], trial_times[3]]])
                isi = (trial_times[2] - trial_times[1])
                
                if trial_labels['trial_type'][ti] == 0:
                    expected = 1000 * mean_long_isi + trial_times[1] 
                elif trial_labels['trial_type'][ti] == 1:
                    expected = 1000 * mean_short_isi + trial_times[1]
                else:
                    expected = np.nan
            else:
                stim_seq = np.array([[np.nan, np.nan], [np.nan, np.nan]])
                isi = np.nan
                expected = np.nan
            
            trial_stim_seq.append(stim_seq)
            trial_isi.append(isi)
            expected_stim.append(expected)
    else:
        # Use bpod event timing
        for ti in range(n_trials):
            if ('BNC1High' in trial_events[ti].keys() and
                'BNC1Low' in trial_events[ti].keys() and
                len(np.array(trial_events[ti]['BNC1High']).reshape(-1)) == 2 and
                len(np.array(trial_events[ti]['BNC1Low']).reshape(-1)) == 2):
                
                stim_seq = 1000 * np.array([trial_events[ti]['BNC1High'], trial_events[ti]['BNC1Low']]) + trial_labels['time_trial_start'][ti]
                stim_seq = np.transpose(stim_seq, [1, 0])
                isi = 1000 * np.array(trial_events[ti]['BNC1High'][1] - trial_events[ti]['BNC1Low'][0])
            else:
                stim_seq = np.array([[np.nan, np.nan], [np.nan, np.nan]])
                isi = np.nan

            if trial_labels['trial_type'][ti] == 0:
                expected = 1000 * np.array(mean_long_isi + trial_events[ti]['BNC1Low'][0]) + trial_labels['time_trial_start'][ti]
            elif trial_labels['trial_type'][ti] == 1:
                expected = 1000 * np.array(mean_short_isi + trial_events[ti]['BNC1Low'][0]) + trial_labels['time_trial_start'][ti]
            else:
                expected = np.nan

            trial_stim_seq.append(stim_seq)
            trial_isi.append(isi)
            expected_stim.append(expected)

    trial_labels['stim_seq'] = np.array(trial_stim_seq + ['yicong_forever'], dtype='object')[:-1]
    trial_labels['isi'] = np.array(trial_isi + ['yicong_forever'], dtype='object')[:-1]
    trial_labels['expected_stim'] = np.array(expected_stim + ['yicong_forever'], dtype='object')[:-1]

    # Licking
    trial_lick = []
    for ti in range(n_trials):
        licking_events = []
        direction = []
        correctness = []
        
        if 'Port1In' in trial_events[ti].keys():
            lick_left = np.array(trial_events[ti]['Port1In']).reshape(-1)
            licking_events.append(lick_left)
            direction.append(np.zeros_like(lick_left))
            if trial_labels['trial_type'][ti] == 0:
                correctness.append(np.ones_like(lick_left))
            else:
                correctness.append(np.zeros_like(lick_left))
        
        if 'Port3In' in trial_events[ti].keys():
            lick_right = np.array(trial_events[ti]['Port3In']).reshape(-1)
            licking_events.append(lick_right)
            direction.append(np.ones_like(lick_right))
            if trial_labels['trial_type'][ti] == 1:
                correctness.append(np.ones_like(lick_right))
            else:
                correctness.append(np.zeros_like(lick_right))
        
        if len(licking_events) > 0:
            licking_events = 1000 * np.concatenate(licking_events).reshape(1, -1) + trial_labels['time_trial_start'][ti]
            direction = np.concatenate(direction).reshape(1, -1)
            correctness = np.concatenate(correctness).reshape(1, -1)
            lick = np.concatenate([licking_events, direction, correctness], axis=0)
            lick = lick[:, np.argsort(lick[0, :])]
            lick = lick[:, lick[0, :] >= trial_labels['state_window_choice'][ti][0]]
            
            if np.size(lick) != 0:
                lick_type = np.full(lick.shape[1], np.nan)
                lick_type[0] = 1
                if (not np.isnan(trial_labels['state_reward'][ti][1]) and
                    len(lick_type) > 1):
                    lick_type[1:][lick[0, 1:] > trial_labels['state_reward'][ti][0]] = 0
                lick_type = lick_type.reshape(1, -1)
                lick = np.concatenate([lick, lick_type], axis=0)
            else:
                lick = np.array([[np.nan], [np.nan], [np.nan], [np.nan]])
        else:
            lick = np.array([[np.nan], [np.nan], [np.nan], [np.nan]])
        
        trial_lick.append(lick)

    trial_labels['lick'] = np.array(trial_lick + ['yicong_forever'], dtype='object')[:-1]

    # Convert to DataFrame
    trial_labels = pd.DataFrame(trial_labels)
    return trial_labels

In [11]:
# remove trial start trigger voltage impulse.
def remove_start_impulse(vol_time, vol_stim_vis):
    min_duration = 100
    changes = np.diff(vol_stim_vis.astype(int))
    start_indices = np.where(changes == 1)[0] + 1
    end_indices = np.where(changes == -1)[0] + 1
    if vol_stim_vis[0] == 1:
        start_indices = np.insert(start_indices, 0, 0)
    if vol_stim_vis[-1] == 1:
        end_indices = np.append(end_indices, len(vol_stim_vis))
    for start, end in zip(start_indices, end_indices):
        duration = vol_time[end-1] - vol_time[start]
        if duration < min_duration:
            vol_stim_vis[start:end] = 0
    return vol_stim_vis

# correct beginning vol_stim_vis if not start from 0.
def correct_vol_start(vol_stim_vis):
    if vol_stim_vis[0] == 1:
        vol_stim_vis[:np.where(vol_stim_vis==0)[0][0]] = 0
    return vol_stim_vis

# detect the rising edge and falling edge of binary series.
def get_trigger_time(
        vol_time,
        vol_bin
        ):
    # find the edge with np.diff and correct it by preappend one 0.
    diff_vol = np.diff(vol_bin, prepend=0)
    idx_up = np.where(diff_vol == 1)[0]
    idx_down = np.where(diff_vol == -1)[0]
    # select the indice for risging and falling.
    # give the edges in ms.
    time_up   = vol_time[idx_up]
    time_down = vol_time[idx_down]
    return time_up, time_down

# find when bpod session timer start.
def get_session_start_time(vol_time, vol_start):
    time_up, _ = get_trigger_time(vol_time, vol_start)
    session_start_time = time_up[0]
    return session_start_time

# correct the fluorescence signal timing.
def correct_time_img_center(time_img):
    # find the frame internal.
    diff_time_img = np.diff(time_img, append=0)
    # correct the last element.
    diff_time_img[-1] = np.mean(diff_time_img[:-1])
    # move the image timing to the center of photon integration interval.
    diff_time_img = diff_time_img / 2
    # correct each individual timing.
    time_neuro = time_img + diff_time_img
    return time_neuro

# save trial neural data.
def save_trials(
        ops, time_neuro, dff, trial_labels,
        vol_time, vol_stim_vis,
        vol_stim_aud, vol_flir,
        vol_pmt, vol_led
        ):
    # file structure:
    # ops['save_path0'] / neural_trials.h5
    # ---- time
    # ---- stim
    # ---- dff
    # ---- vol_stim
    # ---- vol_time
    # trial_labels.csv
    h5_path = os.path.join(ops['save_path0'], 'neural_trials.h5')
    if os.path.exists(h5_path):
        os.remove(h5_path)
    f = h5py.File(h5_path, 'w')
    grp = f.create_group('neural_trials')
    grp['time']         = time_neuro
    grp['dff']          = dff
    grp['vol_time']     = vol_time
    grp['vol_stim_vis'] = vol_stim_vis
    grp['vol_stim_aud'] = vol_stim_aud
    grp['vol_flir']     = vol_flir
    grp['vol_pmt']      = vol_pmt
    grp['vol_led']      = vol_led
    f.close()
    trial_labels.to_csv(os.path.join(ops['save_path0'], 'trial_labels.csv'))

In [10]:
for ops in list_ops:

    print('--- Processing session ---')
    print('Processing session data in:')
    print(ops['save_path0'])
    print('-------------------------------')

    print('Reading dff traces and voltage recordings')
    dff = read_dff(ops)
    [vol_time, vol_start, vol_stim_vis, vol_img,
        vol_hifi, vol_stim_aud, vol_flir,
        vol_pmt, vol_led] = read_raw_voltages(ops)
    vol_stim_vis = remove_start_impulse(vol_time, vol_stim_vis)
    vol_stim_vis = correct_vol_start(vol_stim_vis)
    session_start_time = get_session_start_time(vol_time, vol_start)
    trial_labels = read_bpod_mat_data(ops, session_start_time, vol_time, vol_start, vol_stim_vis)
    print('Correcting 2p camera trigger time')
    # signal trigger time stamps.
    time_img, _   = get_trigger_time(vol_time, vol_img)
    # correct imaging timing.
    time_neuro = correct_time_img_center(time_img)
    # save the final data.
    print('Saving trial data')
    save_trials(
        ops, time_neuro, dff, trial_labels,
        vol_time, vol_stim_vis,
        vol_stim_aud, vol_flir,
        vol_pmt, vol_led)


--- Processing session ---
Processing session data in:
/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/2p/YH24LG_CRBL_lobulev_20250814_2afc-584
-------------------------------
Reading dff traces and voltage recordings
Correcting 2p camera trigger time
Saving trial data


In [8]:
from scipy.signal import savgol_filter

# read trial label csv file into dataframe.
def read_trial_label(ops):
    raw_csv = pd.read_csv(os.path.join(ops['save_path0'], 'trial_labels.csv'), index_col=0)
    # recover object numpy array from csv str.
    def object_parse(k, shape):
        arr = np.array(
            [np.fromstring(s.replace('[', '').replace(']', ''), sep=' ').reshape(shape)
             for s in raw_csv[k].to_list()] + ['yicong_forever'],
            dtype='object')[:-1]
        return arr
    # parse all array.
    time_trial_start = raw_csv['time_trial_start'].to_numpy(dtype='float32')
    time_trial_end = raw_csv['time_trial_end'].to_numpy(dtype='float32')
    trial_type = raw_csv['trial_type'].to_numpy(dtype='int8')
    block_type = raw_csv['block_type'].to_numpy(dtype='int8')
    outcome = raw_csv['outcome'].to_numpy(dtype='object')
    state_window_choice = object_parse('state_window_choice', [-1])
    state_reward = object_parse('state_reward', [-1])
    state_punish = object_parse('state_punish', [-1])
    stim_seq = object_parse('stim_seq', [-1,2])
    expected_stim = object_parse('expected_stim', [-1])
    isi = raw_csv['isi'].to_numpy(dtype='float32')
    lick = object_parse('lick', [4,-1])
    # convert to dataframe.
    trial_labels = pd.DataFrame({
        'time_trial_start': time_trial_start,
        'time_trial_end': time_trial_end,
        'trial_type': trial_type,
        'block_type': block_type,
        'outcome': outcome,
        'state_window_choice': state_window_choice,
        'state_reward': state_reward,
        'state_punish': state_punish,
        'stim_seq': stim_seq,
        'isi': isi,
        'expected_stim': expected_stim,
        'lick': lick,
        })
    return trial_labels

def zscore_normalize(data, axis=1, min_std=1e-8):
    """
    Perform Z-score normalization on 2P neural data.
    
    Parameters:
    - data: NumPy array (n_neurons x n_timepoints) or 1D array for single trace
    - axis: Axis along which to compute mean and std (default: 1 for time axis)
    - min_std: Minimum standard deviation to avoid division by zero
    
    Returns:
    - z_data: Z-score normalized data (same shape as input)
    """
    # Ensure input is a NumPy array
    data = np.asarray(data)
    
    # Handle 1D input (single trace) by reshaping
    if data.ndim == 1:
        data = data.reshape(1, -1)
    
    # Compute mean and standard deviation along specified axis
    mean = np.mean(data, axis=axis, keepdims=True)
    std = np.std(data, axis=axis, keepdims=True)
    
    # Prevent division by zero by setting a minimum std
    std = np.maximum(std, min_std)
    
    # Z-score normalization: (data - mean) / std
    z_data = (data - mean) / std
    
    return z_data

# read trailized neural traces with stimulus alignment.
def read_neural_trials(ops, smooth):
    mm_path, file_path = get_memmap_path(ops, 'neural_trials.h5')
    trial_labels = read_trial_label(ops)
    with h5py.File(file_path, 'r') as f:
        neural_trials = dict()
        dff = np.array(f['neural_trials']['dff'])
        dff = zscore_normalize(dff)
        if smooth:
            window_length=5
            polyorder=3
            dff = np.apply_along_axis(
                savgol_filter, 1, dff,
                window_length=window_length,
                polyorder=polyorder)
        else: pass
        neural_trials['dff']          = create_memmap(dff,                                'float32', os.path.join(mm_path, 'dff.mmap'))
        neural_trials['time']         = create_memmap(f['neural_trials']['time'],         'float32', os.path.join(mm_path, 'time.mmap'))
        neural_trials['trial_labels'] = trial_labels
        neural_trials['vol_time']     = create_memmap(f['neural_trials']['vol_time'],     'float32', os.path.join(mm_path, 'vol_time.mmap'))
        neural_trials['vol_stim_vis'] = create_memmap(f['neural_trials']['vol_stim_vis'], 'int8',    os.path.join(mm_path, 'vol_stim_vis.mmap'))
        neural_trials['vol_stim_aud'] = create_memmap(f['neural_trials']['vol_stim_aud'], 'float32', os.path.join(mm_path, 'vol_stim_aud.mmap'))
        neural_trials['vol_flir']     = create_memmap(f['neural_trials']['vol_flir'],     'int8',    os.path.join(mm_path, 'vol_flir.mmap'))
        neural_trials['vol_pmt']      = create_memmap(f['neural_trials']['vol_pmt'],      'int8',    os.path.join(mm_path, 'vol_pmt.mmap'))
        neural_trials['vol_led']      = create_memmap(f['neural_trials']['vol_led'],      'int8',    os.path.join(mm_path, 'vol_led.mmap'))
    return neural_trials

In [9]:
import pandas as pd
list_neural_trials = []
for ops in list_ops:
    print('--- Processing session ---')
    print('Processing session data in:')
    print(ops['save_path0'])
    print('-------------------------------')
    print('Reading trailized neural traces with stimulus alignment')
    neural_trials = read_neural_trials(ops, 0)
    list_neural_trials.append(neural_trials)


--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250814_2afc-584
-------------------------------
Reading trailized neural traces with stimulus alignment


# LFAD

## alginnment and data preparation

In [30]:
from tqdm import tqdm
import numpy as np
import h5py
import os
from scipy.interpolate import interp1d

def trim_seq(data, pivots):
    """
    Cuts a list of sequences (or 3D arrays) to the same length based on pivot points.
    
    The function finds the minimum available history (left of pivot) and 
    minimum available future (right of pivot) across all trials to ensure 
    no padding is required.
    
    Args:
        data (list): List of arrays. Can be 1D (time,) or 3D (channels, time).
        pivots (list/array): Indices in each sequence representing the alignment event (t=0).
        
    Returns:
        list: The trimmed data where every trial has the same time dimension.
    """
    # CASE 1: 1D Data (e.g., time vectors, behavioral traces)
    if len(data[0].shape) == 1:
        # Find shortest distance from start to pivot across all trials
        len_l_min = np.min(pivots)
        # Find shortest distance from pivot to end across all trials
        len_r_min = np.min([len(data[i]) - pivots[i] for i in range(len(data))])
        
        # Slice every trial to these minimum boundaries
        data = [data[i][pivots[i] - len_l_min : pivots[i] + len_r_min]
                for i in range(len(data))]

    # CASE 2: 3D Data (e.g., Neural Data: [1, Neurons, Time])
    if len(data[0].shape) == 3:
        # Find shortest distance from start to pivot
        len_l_min = np.min(pivots)
        # Find shortest distance from pivot to end (assuming time is axis 2)
        len_r_min = np.min([len(data[i][0, 0, :]) - pivots[i] for i in range(len(data))])
        
        # Slice the time dimension (axis 2)
        data = [data[i][:, :, pivots[i] - len_l_min : pivots[i] + len_r_min]
                for i in range(len(data))]
    
    return data

def get_perception_response(neural_trials, target_state, l_frames, r_frames, indices=0):
    """
    Aligns neural data to a specific trial state (e.g., 'stim_start', 'outcome').
    
    Args:
        neural_trials (dict): Data dictionary.
        target_state (str): Key in 'trial_labels' to align to (e.g., 'stim_start').
        l_frames (int): Frames before alignment point.
        r_frames (int): Frames after alignment point.
        indices (int): Sub-index if the state has multiple values (default 0).
    """
    exclude_start_trials = 2
    exclude_end_trials = 2
    
    # --- Initialization ---
    time = neural_trials['time']
    # Neural data containers
    neu_seq    = []
    neu_time   = []
    
    # External Input / Behavioral containers
    stim_seq   = [] # Stimulus sequence timing
    stim_value = [] # Voltage value of stimulus
    stim_time  = [] # Time vector for voltage
    led_value  = [] # LED indicator
    
    # Task metadata containers
    trial_type = []
    block_type = []
    isi        = []
    decision   = []
    outcome    = []

    # --- Loop over trials ---
    total_trials = len(neural_trials['trial_labels'])
    for ti in tqdm(range(total_trials), desc="Processing Trials"):
        
        # Get alignment time 't' for the specific state
        t = neural_trials['trial_labels'][target_state][ti].flatten()[indices]
        
        # Check Validity: Time exists AND trial is not in exclusion zone
        if (not np.isnan(t) and
            ti >= exclude_start_trials and
            ti < total_trials - exclude_end_trials
            ):
            
            # Find alignment index
            idx = np.searchsorted(neural_trials['time'], t)
            
            # Boundary Check
            if idx > l_frames and idx < len(neural_trials['time']) - r_frames:
                
                # --- Neural Data Extraction ---
                f = neural_trials['dff'][:, idx - l_frames : idx + r_frames]
                f = np.expand_dims(f, axis=0)
                neu_seq.append(f)
                
                # Relative Neural Time
                neu_time.append(neural_trials['time'][idx - l_frames : idx + r_frames] - time[idx])
                
                # --- Voltage/Stimulus Data Extraction ---
                # Voltage usually has a different sampling rate, so we map indices separately
                vol_t_c = np.searchsorted(neural_trials['vol_time'], neural_trials['time'][idx])
                vol_t_l = np.searchsorted(neural_trials['vol_time'], neural_trials['time'][idx - l_frames])
                vol_t_r = np.searchsorted(neural_trials['vol_time'], neural_trials['time'][idx + r_frames])
                
                stim_time.append(neural_trials['vol_time'][vol_t_l:vol_t_r] - neural_trials['vol_time'][vol_t_c])
                stim_value.append(neural_trials['vol_stim_vis'][vol_t_l:vol_t_r])
                led_value.append(neural_trials['vol_led'][vol_t_l:vol_t_r])
                
                # --- Task Variables ---
                stim_seq.append(neural_trials['trial_labels']['stim_seq'][ti].reshape(1, 2, 2) - t)
                trial_type.append(neural_trials['trial_labels']['trial_type'][ti])
                block_type.append(neural_trials['trial_labels']['block_type'][ti])
                isi.append(neural_trials['trial_labels']['isi'][ti])
                decision.append(neural_trials['trial_labels']['lick'][ti][1, 0])
                outcome.append(neural_trials['trial_labels']['outcome'][ti])
        else: 
            pass

    # --- Post-Extraction Alignment (Trim to Common Shape) ---
    
    # 1. Neural Alignment
    neu_time_zero = [np.argmin(np.abs(nt)) for nt in neu_time]
    neu_time = trim_seq(neu_time, neu_time_zero)
    neu_seq = trim_seq(neu_seq, neu_time_zero)
    
    # 2. Voltage/Stimulus Alignment
    # Voltage data needs its own trimming as it might have slight jitter in sampling
    stim_time_zero = [np.argmin(np.abs(sv)) for sv in stim_value]
    stim_time = trim_seq(stim_time, stim_time_zero)
    stim_value = trim_seq(stim_value, stim_time_zero)
    led_value = trim_seq(led_value, stim_time_zero)

    # --- Concatenation ---
    neu_seq    = np.concatenate(neu_seq, axis=0)        # Shape: [N_Trials, N_Neurons, N_Time]
    neu_time   = [nt.reshape(1, -1) for nt in neu_time]
    neu_time   = np.concatenate(neu_time, axis=0)
    
    stim_seq   = np.concatenate(stim_seq, axis=0)
    
    stim_value = [sv.reshape(1, -1) for sv in stim_value]
    stim_value = np.concatenate(stim_value, axis=0)     # Shape: [N_Trials, N_Vol_Time]
    
    stim_time  = [st.reshape(1, -1) for st in stim_time]
    stim_time  = np.concatenate(stim_time, axis=0)
    
    led_value  = [lv.reshape(1, -1) for lv in led_value]
    led_value  = np.concatenate(led_value, axis=0)
    
    # Metadata arrays
    trial_type = np.array(trial_type)
    block_type = np.array(block_type)
    isi        = np.array(isi)
    decision   = np.array(decision)
    outcome    = np.array(outcome)
    
    # Mean time vectors for visualization
    neu_time  = np.mean(neu_time, axis=0)
    stim_time = np.mean(stim_time, axis=0)

    return [neu_seq, neu_time, stim_seq, stim_value, stim_time, led_value, trial_type, block_type, isi, decision, outcome]

def prepare_data_for_lfads(neural_trials, target_state, l_frames, r_frames, 
                           output_path="lfads_data.h5", 
                           run_name="dataset_01"):
    """
    Prepares aligned data for Gaussian LFADs (continuous dF/F).
    
    Changes from previous version:
    1. Transposes dimensions to (Trials, Time, Neurons).
    2. Saves directly to HDF5.
    
    Args:
        output_path (str): Where to save the .h5 file.
    """
    
    # --- 1. Extract Aligned Data ---
    print(f"[{run_name}] Extracting aligned data for state: {target_state}...")
    extracted_data = get_perception_response(
        neural_trials, target_state, l_frames, r_frames
    )
    
    # Unpack
    neu_seq     = extracted_data[0]   
    neu_time    = extracted_data[1]   
    stim_seq    = extracted_data[2]   
    stim_value  = extracted_data[3]   
    stim_time   = extracted_data[4]   
    led_value   = extracted_data[5]   
    trial_type  = extracted_data[6]   
    block_type  = extracted_data[7]   
    isi         = extracted_data[8]   
    decision    = extracted_data[9]   
    outcome     = extracted_data[10]  

    # --- 2. Transpose for LFADs ---
    print("Transposing data to (Trials, Time, Neurons) for LFADs compatibility...")
    neu_seq = np.transpose(neu_seq, (0, 2, 1))

    # --- 3. Process External Inputs ---
    n_trials, n_time, n_neurons = neu_seq.shape
    lfads_inputs = np.zeros((n_trials, n_time, 1))
    
    print("Interpolating stimulus data...")
    for i in range(n_trials):
        f_interp = interp1d(stim_time, stim_value[i], kind='linear', fill_value="extrapolate")
        lfads_inputs[i, :, 0] = f_interp(neu_time)

    # --- 4. Train/Validation Split ---
    indices = np.arange(n_trials)
    np.random.shuffle(indices)
    
    split_idx = int(0.8 * n_trials)
    train_idxs = indices[:split_idx]
    valid_idxs = indices[split_idx:]
    
    def split_var(arr):
        if arr is None: 
            return None, None
        return arr[train_idxs], arr[valid_idxs]

    # Split Tensors
    train_data, valid_data = split_var(neu_seq)
    train_ext, valid_ext   = split_var(lfads_inputs)
    
    # Split Metadata
    train_dict = {
        'inds': train_idxs,
        'stim_seq': split_var(stim_seq)[0],
        'decision': split_var(decision)[0],
        'outcome':  split_var(outcome)[0],
        'trial_type': split_var(trial_type)[0]
    }
    
    valid_dict = {
        'inds': valid_idxs,
        'stim_seq': split_var(stim_seq)[1],
        'decision': split_var(decision)[1],
        'outcome':  split_var(outcome)[1],
        'trial_type': split_var(trial_type)[1]
    }

    # --- 5. Helper: Convert strings to HDF5-compatible format ---
    def h5_compatible(data):
        """Converts Unicode strings to ASCII bytes for HDF5."""
        if data is None:
            return data
        if data.dtype.kind == 'U':  # Unicode string
            return data.astype('S')  # Convert to byte string
        return data

    # --- 6. Save to HDF5 ---
    print(f"Saving data to {output_path}...")
    
    with h5py.File(output_path, 'w') as hf:
        # A. Required Tensors for LFADs
        hf.create_dataset('train_data', data=train_data)
        hf.create_dataset('valid_data', data=valid_data)
        hf.create_dataset('train_ext_input', data=train_ext)
        hf.create_dataset('valid_ext_input', data=valid_ext)
        
        # B. Saving Metadata (with string conversion)
        g_train = hf.create_group('train_meta')
        for k, v in train_dict.items():
            g_train.create_dataset(k, data=h5_compatible(v))
            
        g_valid = hf.create_group('valid_meta')
        for k, v in valid_dict.items():
            g_valid.create_dataset(k, data=h5_compatible(v))
            
        # C. Global Info
        hf.create_dataset('dt', data=np.mean(np.diff(neu_time)))
        hf.create_dataset('neu_time_vec', data=neu_time)

    print(f"Process Complete.")
    print(f"Train Data Shape: {train_data.shape} (Trials, Time, Neurons)")
    print(f"Valid Data Shape: {valid_data.shape}")
    print(f"Output saved to: {os.path.abspath(output_path)}")
    
    return train_data, valid_data
# Usage Example:
prepare_data_for_lfads(neural_trials, 'stim_seq', 30, 150, 
                           output_path="lfads_data.h5", 
                           run_name="dataset_01")

[dataset_01] Extracting aligned data for state: stim_seq...


Processing Trials: 100%|██████████| 403/403 [00:00<00:00, 3874.19it/s]


Transposing data to (Trials, Time, Neurons) for LFADs compatibility...
Interpolating stimulus data...
Saving data to lfads_data.h5...
Process Complete.
Train Data Shape: (319, 180, 253) (Trials, Time, Neurons)
Valid Data Shape: (80, 180, 253)
Output saved to: f:\Single_Interval_discrimination\Code\2p\2p_2AFC_double_block_version\Test_pilot\lfads_data.h5


(array([[[ 1.78482428e-01,  2.13512397e+00,  1.18016921e-01, ...,
           3.94653976e-01, -3.62823270e-02, -5.13102829e-01],
         [ 9.11211848e-01,  1.92938864e+00,  2.11163449e+00, ...,
          -3.27257738e-02,  2.85180002e-01,  4.45467329e+00],
         [ 3.53425324e-01, -7.66439214e-02,  1.08490682e+00, ...,
           1.86676562e-01,  5.89859247e-01, -2.50495720e+00],
         ...,
         [ 6.18227780e-01, -2.03779802e-01, -6.52541459e-01, ...,
          -3.41360122e-01, -1.82480621e+00, -1.25512981e+00],
         [ 6.20932877e-01,  2.94580579e-01,  1.83105910e+00, ...,
           8.42360616e-01, -1.15232003e+00, -2.44349957e+00],
         [ 2.91585475e-01,  2.47764778e+00, -8.20709944e-01, ...,
           1.04023311e-02, -4.51547325e-01, -1.41717267e+00]],
 
        [[ 1.48153889e+00, -1.06311154e+00, -1.15000784e+00, ...,
          -1.22235286e+00, -1.03613746e+00,  3.53590325e-02],
         [ 1.06824410e+00, -2.47751474e-01, -1.23406231e+00, ...,
           1.21992707

In [32]:
# Alternative if the command above fails
import sys
!{sys.executable} -m lfads-torch.lfads_torch.run_model --config-name config --config-dir .

In [ ]:
import sys
!{sys.executable} -m lfads-torch.train --config-name config --config-dir .

f:\Single_Interval_discrimination\Code\2p\2p_2AFC_double_block_version\Test_pilot\.venv\Scripts\python.exe: No module named lfads-torch.train


: 